# 🧙‍♂️ Counsellor POC — Decision Archaeology with a Self-Evolving KBA

Connecting Dots... 👉 https://sriharshacr.github.io/blogs/

## What This Notebook Shows

A minimal, runnable proof-of-concept for **Counsellor** — an AI agent that maintains a Laws store
and governs decisions across a multi-agent system.

**The story, in one run:**
1. Memory hits 91% — above the trigger in the Law of Memory Management
2. Counsellor activates `agent-memory-sniffer`
3. The sniffer analyses entries, generates a structured remediation plan (one LLM call)
4. A GitHub Issue is created for review (simulated — see the swap note at the end)
5. You approve → stale entries auto-removed → memory drops below 80%

**One dependency:** `groq` (free key at [console.groq.com](https://console.groq.com))

> 🔌 **GitHub integration is simulated.** Look for `# 🔌 SWAP` comments to wire up real GitHub Issues.

## How to Interact With This Notebook

You do not need to write any code to follow this notebook. Run the cells top to bottom — each one builds on the previous.

**There is one decision point, in Cell 12:** an `approved` flag that simulates your review of the GitHub Issue. Set it to `True` to let the cleanup proceed. Set it to `False` to see what the system looks like when a review is still pending — the Issue exists, the plan is ready, but nothing happens until you approve.

Everything else is observable, not configurable. You will see:
- what memory looks like before Counsellor runs (Cell 8)
- what the Law detects and why it fires (Cell 9)
- what the AI agent recommends after analysing the entries (Cell 10)
- what a GitHub Issue would look like (Cell 11)
- what the memory looks like after cleanup (Cell 14)

**Want to experiment further?** The two most useful swaps are:
- **Cell 7** — replace the placeholder memory entries with your own content. The structure stays the same; only the values change.
- **Cell 5** — swap the Groq API key and model. Any OpenAI-compatible provider works; the swap comment shows you how.

In [ ]:
# One dependency
!pip install groq -q
import json
from groq import Groq
from google.colab import userdata

print('✅ Ready')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────────
# 🔌 SWAP: Replace with any OpenAI-compatible provider
#   OpenAI:  from openai import OpenAI; client = OpenAI(api_key='...')
#   Ollama:  Groq(api_key='ollama', base_url='http://localhost:11434/v1')

GROQ_API_KEY = userdata.get('GROQ_API_KEY')  # Free: console.groq.com
LLM_MODEL    = 'qwen/qwen3.8-27b'
client       = Groq(api_key=GROQ_API_KEY)

MEMORY_LIMIT      = 2200   # Hard limit in chars (Hermes default)
TRIGGER_THRESHOLD = 0.90   # Law fires above this %
HARD_CAP          = 0.95   # System rejects new memories above this %
TARGET_USAGE      = 0.80   # Post-cleanup target

print(f'Config loaded')
print(f'  Limit={MEMORY_LIMIT}c | Trigger>{TRIGGER_THRESHOLD:.0%} | Hard cap>{HARD_CAP:.0%} | Target<{TARGET_USAGE:.0%}')

In [ ]:
# ── Laws Store ───────────────────────────────────────────────────────────────────
# In production: swap dict for ChromaDB or SQLite for semantic retrieval over Laws
# 🔌 SWAP: chromadb.Client() → collection.add(documents=[law_text], ids=[law_id])

LAWS = {
    'law_of_memory_management': {
        'name'     : 'Law of Memory Management',
        'trigger'  : 'memory_pct > 0.90',
        'agent'    : 'agent-memory-sniffer',
        'auto_rule': 'age_days > 15 AND relevance_days < 7',
        'target'   : TARGET_USAGE,
        'by'       : 'Counsellor 🧙‍♂️',
    },
    'law_of_draft_lifecycle': {
        'name'  : 'Law of Draft Lifecycle',
        'trigger': 'draft_age_days > 7',
        'agent' : 'agent-draft-janitor',
        'rule'  : 'local draft → GitHub Discussion → delete local (TTL=7d)',
        'by'    : 'Counsellor 🧙‍♂️',
    },
}

print(f'📜 Laws Store: {len(LAWS)} Laws registered')
for law in LAWS.values():
    print(f"  {law['name']}  →  triggers: {law['agent']}")

## Understanding Memory Entries

Before looking at the entries, here is what each field means — because the agent's entire decision about what to keep, flag, or remove is based on reading these fields.

**`content`** — The actual knowledge stored. This is what gets retrieved when Counsellor or another agent needs context before making a decision. It can be a learned lesson, a behavioral constraint, a process step, or a personal preference. The content should be self-contained — it needs to make sense without any surrounding context, because it may be read months later by an agent that has no memory of how it was created.

**`category`** — The type of knowledge. This tells the agent how aggressively it can clean the entry:
| Category | What it stores | Auto-remove eligible? |
|---|---|---|
| `strategic` | Timeless principles — architectural decisions, design philosophy | Never |
| `lesson` | Learned from an incident or mistake | Never |
| `profile` | User preferences, locale, working style, system pointers | Never |
| `operational` | Process instructions — checklists, protocols, workflows | Only if stale by threshold |
| `tactical` | Short-term, time-bound facts — deadlines, blockers, reminders | Yes, aggressively |

**`relevance_days`** — How long this entry is expected to remain useful. A tactical reminder about a meeting next week might be relevant for 3 days. A strategic architectural principle is relevant for 365. This is not set automatically — it is declared when the entry is created, as a commitment about the entry's expected useful life.

**`age_days`** — How many days ago this entry was written. The agent combines this with `relevance_days` to detect entries that have outlived their usefulness: if an entry is older than it was ever meant to be relevant, it is a cleanup candidate.

**The auto-remove rule** is simple: `age_days > 15 AND relevance_days < 7`. An old tactical entry — past its expected useful life — gets removed automatically. No human input needed. Anything that does not clearly match this rule gets flagged for your decision instead.

In [ ]:
# ── Simulated Memory State ────────────────────────────────────────────────────
# Realistic placeholder entries. Swap with your own MEMORY.md content.
# Fields: key | content | category | relevance_days | age_days
#
# Categories:
#   strategic   — timeless principles, never auto-remove
#   lesson      — learned from experience, long-term value
#   profile     — user/agent preferences, permanent
#   operational — process instructions, medium-term
#   tactical    — short-term facts, stale candidates

MEMORY = [
    # ─ Strategic (permanent) ─────────────────────────────────────────
    {'key': 's1', 'content': 'Architectural Protocol: prefer composition over inheritance. Validate at system boundaries, trust internals. Complexity is a cost that must be justified.', 'category': 'strategic', 'relevance_days': 365, 'age_days': 45},
    {'key': 's2', 'content': 'Simplicity Definition: a solution is simple when removing any part breaks it. Never add abstractions ahead of need.', 'category': 'strategic', 'relevance_days': 365, 'age_days': 30},
    # ─ Lessons (long-term) ───────────────────────────────────────────
    {'key': 'l1', 'content': 'Never update shared config files without notifying all dependent services. An email service outage was traced to an unannounced config change.', 'category': 'lesson', 'relevance_days': 180, 'age_days': 20},
    {'key': 'l2', 'content': 'Always version-lock external dependencies in agent workflows. Upstream changes broke the summariser skill twice in one month.', 'category': 'lesson', 'relevance_days': 180, 'age_days': 18},
    {'key': 'l3', 'content': 'Agents that write to disk need explicit TTL policies. Without them, temp files accumulate silently and cause disk pressure over weeks.', 'category': 'lesson', 'relevance_days': 180, 'age_days': 12},
    {'key': 'l4', 'content': 'Never deploy on Fridays or before public holidays. Two incidents were traced to weekend deployments with no on-call coverage.', 'category': 'lesson', 'relevance_days': 180, 'age_days': 8},
    {'key': 'l5', 'content': 'Service-to-service calls must include correlation IDs for distributed tracing. Debugging a cross-service incident took 4 hours due to missing trace context.', 'category': 'lesson', 'relevance_days': 180, 'age_days': 6},
    # ─ Profile (permanent) ───────────────────────────────────────────
    {'key': 'p1', 'content': 'User prefers direct communication, no padding. Values evidence over opinion. Batch questions, no rapid-fire. Summarise decisions at end of session.', 'category': 'profile', 'relevance_days': 365, 'age_days': 60},
    {'key': 'p2', 'content': 'Locale: IST (UTC+5:30). Working hours 9AM-7PM IST. Peak focus 10AM-1PM and 8PM-11PM. Async-first outside those windows.', 'category': 'profile', 'relevance_days': 365, 'age_days': 60},
    {'key': 'p3', 'content': 'KBA System pointer: github.com/counsellor-demo/kba - Laws, Decisions, Patterns in GitHub Discussions. Memory stores URL pointer only.', 'category': 'profile', 'relevance_days': 365, 'age_days': 5},
    # ─ Operational (medium-term) ──────────────────────────────────────
    {'key': 'o1', 'content': 'Deployment checklist: run tests locally, push to staging, validate metrics, get approval, deploy prod, monitor for 30 mins.', 'category': 'operational', 'relevance_days': 90, 'age_days': 25},
    {'key': 'o2', 'content': 'Code review: read PR description, check tests, review logic, check edge cases, comment with reasoning, approve or request changes.', 'category': 'operational', 'relevance_days': 90, 'age_days': 30},
    {'key': 'o3', 'content': 'Meeting protocol: set agenda 24h before, timeboxed items, log decisions, assign action items with owners and dates, follow up within 48h.', 'category': 'operational', 'relevance_days': 90, 'age_days': 22},
    {'key': 'o4', 'content': 'Incident response: open GitHub issue with incident label, assign on-call, post status every 30 mins, conduct postmortem within 48 hours.', 'category': 'operational', 'relevance_days': 90, 'age_days': 15},
    # ─ Tactical (STALE — auto-remove candidates) ───────────────────────
    {'key': 't1', 'content': 'Active trial: shared services POC deadline was Oct 15. Review meeting Oct 17. Outcome: approved for phase 2.', 'category': 'tactical', 'relevance_days': 3, 'age_days': 22},
    {'key': 't2', 'content': 'Blocker Oct 20: waiting for infra team to provision staging namespace. ETA 2 days. Follow up if not resolved by Oct 22.', 'category': 'tactical', 'relevance_days': 2, 'age_days': 18},
    {'key': 't3', 'content': 'Reminder Oct 28: weekly memory review. Agenda - check trial progress, archive completed items, update KBA pointers.', 'category': 'tactical', 'relevance_days': 1, 'age_days': 16},
]

total = sum(len(e['content']) for e in MEMORY)
pct   = total / MEMORY_LIMIT
print(f'Memory loaded: {len(MEMORY)} entries | {total}/{MEMORY_LIMIT} chars ({pct:.1%})')
print()
from collections import Counter
cats = Counter(e['category'] for e in MEMORY)
for cat, n in sorted(cats.items()):
    cat_chars = sum(len(e['content']) for e in MEMORY if e['category'] == cat)
    print(f'  {cat:<12} {n} entries  {cat_chars} chars')

In [ ]:
# ── BEFORE: No Law Active ─────────────────────────────────────────────────────
status = '🔴 CRITICAL' if pct >= HARD_CAP else '🟡 WARNING' if pct >= TRIGGER_THRESHOLD else '🟢 OK'

print('=' * 55)
print('  BEFORE — No Counsellor Law active')
print('=' * 55)
print(f'  Memory:  {total}/{MEMORY_LIMIT} chars  ({pct:.1%})  {status}')
print()
print('  Without Counsellor:')
print('  ✗  Memory keeps filling silently')
print(f'  ✗  At {HARD_CAP:.0%}: new memories rejected')
print('  ✗  Learnings from this session may not persist')
print('  ✗  Cleanup only happens when someone notices')

In [ ]:
# ── Counsellor: Check Laws ──────────────────────────────────────────────────
print('🧙‍♂️ Counsellor scanning Laws...\n')

triggered = None
for law_id, law in LAWS.items():
    if law_id == 'law_of_memory_management' and pct > TRIGGER_THRESHOLD:
        triggered = law
        print(f"  🔴 {law['name']} triggered")
        print(f"     Condition: {pct:.1%} > {TRIGGER_THRESHOLD:.0%}")
        print(f"     Activating: `{law['agent']}`")
        break

if not triggered:
    print('  ✅ All Laws satisfied. No action required.')

In [ ]:
# ── agent-memory-sniffer: One LLM Call ──────────────────────────────────────
# This is the AI doing real work: analyse entries, produce a structured plan.

memory_summary = '\n'.join(
    f"  [{e['key']}] {e['category']} | age={e['age_days']}d | relevance={e['relevance_days']}d\n  content: {e['content'][:80]}..."
    for e in MEMORY
)

prompt = (
    f'You are agent-memory-sniffer, activated by Counsellor under the Law of Memory Management.\n'
    f'\n'
    f'CURRENT STATE\n'
    f'Usage: {total}/{MEMORY_LIMIT} chars ({pct:.1%})\n'
    f'Trigger: >{TRIGGER_THRESHOLD:.0%} | Hard cap: >{HARD_CAP:.0%} | Target: <{TARGET_USAGE:.0%}\n'
    f'\n'
    f'AUTO-REMOVE RULE (no human input needed)\n'
    f'Remove entries where: age_days > 15 AND relevance_days < 7\n'
    f'NEVER auto-remove categories: strategic, profile, lesson\n'
    f'\n'
    f'MEMORY ENTRIES\n'
    f'{memory_summary}\n'
    f'\n'
    f'Return valid JSON with exactly these keys:\n'
    f'- title (string): short issue title\n'
    f'- problem (string): one sentence describing the problem\n'
    f'- impact (list of strings): 2-3 impact items\n'
    f'- root_cause (string): one sentence\n'
    f'- auto_remove (list of strings): entry keys matching the auto-remove rule only\n'
    f'- flag_for_human (list of strings): entry keys needing human decision, format "key | reason"\n'
    f'- keep (list of strings): entry keys that are safe and should stay\n'
    f'- est_savings_chars (integer): estimated chars freed by auto-remove\n'
)

print('🔍 agent-memory-sniffer analysing entries...\n')

resp   = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{'role': 'user', 'content': prompt}],
    temperature=0.1,
    response_format={'type': 'json_object'},
)
report = json.loads(resp.choices[0].message.content)

print('✅ Analysis complete')
print(f"  Auto-remove: {report['auto_remove']}")
print(f"  Flag:        {[f.split('|')[0].strip() for f in report['flag_for_human']]}")
print(f"  Keep:        {len(report['keep'])} entries")
print(f"  Est. savings: ~{report['est_savings_chars']} chars")

In [ ]:
# ── Simulated GitHub Issue ─────────────────────────────────────────────────
# 🔌 SWAP: Replace this print block with real GitHub API:
#
#   from github import Github
#   g     = Github(os.environ['GITHUB_TOKEN'])
#   repo  = g.get_repo('your-username/counsellor-kba')
#   issue = repo.create_issue(
#       title    = report['title'],
#       body     = issue_body,
#       assignees= ['your-username'],
#       labels   = ['memory-management', 'counsellor-auto'],
#   )
#   print(f'Issue created: {issue.html_url}')

est_saved   = sum(len(e['content']) for e in MEMORY if e['key'] in report['auto_remove'])
impact_str  = '\n'.join('- ' + i for i in report['impact'])
auto_str    = '\n'.join('- `' + k + '`' for k in report['auto_remove'])
flag_str    = '\n'.join('- ' + f for f in report['flag_for_human'])

issue_body = (
    f"**Flagged by:** `agent-memory-sniffer`\n"
    f"**Law:** Law of Memory Management | **Assigned to:** @you\n"
    f"\n---\n"
    f"### Problem\n{report['problem']}\n"
    f"\n### Impact\n{impact_str}\n"
    f"\n### Root Cause\n{report['root_cause']}\n"
    f"\n---\n"
    f"### Proposed Actions\n"
    f"\n**✅ Auto-remove** (age > 15d AND relevance < 7d — no approval needed):\n"
    f"{auto_str}\n"
    f"*Estimated savings: ~{est_saved} chars*\n"
    f"\n**⚠️ Flag for your review:**\n"
    f"{flag_str}\n"
    f"\n---\n"
    f"*Maintained by Counsellor 🧙‍♂️*"
)

print('━' * 55)
print('  📋 GITHUB ISSUE  (simulated — see SWAP comment above)')
print('━' * 55)
print(issue_body)
print('━' * 55)

In [ ]:
# ── Your Approval ───────────────────────────────────────────────────────────
# In the real workflow: review the GitHub Issue thread, approve via comment.
# Here: set approved = True and run this cell to proceed.

approved = True   # ← set False to see what happens when review is pending

if approved:
    print('✅ Approved — running auto-cleanup')
else:
    print('⏸  Pending — review the issue above, then set approved = True')

In [ ]:
# ── Auto-cleanup ────────────────────────────────────────────────────────────
if approved:
    remove_keys = set(report['auto_remove'])
    removed     = [e for e in MEMORY if e['key'] in remove_keys]
    cleaned     = [e for e in MEMORY if e['key'] not in remove_keys]
    freed       = sum(len(e['content']) for e in removed)

    print(f'Removed {len(removed)} stale entries ({freed} chars freed):')
    for e in removed:
        print(f"  ❌ [{e['key']}] age={e['age_days']}d | relevance={e['relevance_days']}d | {e['content'][:55]}...")

    if report['flag_for_human']:
        print(f"\n⚠️  {len(report['flag_for_human'])} entries await your decision:")
        for item in report['flag_for_human']:
            key = item.split('|')[0].strip()
            entry = next((e for e in MEMORY if e['key'] == key), None)
            reason = item.split('|')[1].strip() if '|' in item else ''
            if entry:
                print(f"  ? [{key}] {reason} | {entry['content'][:55]}...")
else:
    cleaned = MEMORY
    freed   = 0
    print('No changes applied.')

In [ ]:
# ── AFTER ─────────────────────────────────────────────────────────────────
new_total = sum(len(e['content']) for e in cleaned)
new_pct   = new_total / MEMORY_LIMIT
target_ok = new_pct < TARGET_USAGE

print('=' * 55)
print('  AFTER — Counsellor Law executed')
print('=' * 55)
print(f'  {"":5} {"BEFORE":>22}  {"AFTER":>22}')
print(f'  {"":5} {"-"*22}  {"-"*22}')
print(f'  Usage: {total:>5}/{MEMORY_LIMIT} ({pct:.1%})   →   {new_total:>5}/{MEMORY_LIMIT} ({new_pct:.1%})')
print(f'  Entries: {len(MEMORY):>19}   →   {len(cleaned):>19}')
print()
print(f'  Target (<{TARGET_USAGE:.0%}):  {"\u2705 Met" if target_ok else "\u26a0\ufe0f  Review flagged entries"}')
print(f'  Chars freed:   {freed}')
print()
print('  What just happened:')
print(f'  1. Memory hit {pct:.1%}  →  Law of Memory Management triggered')
print( '  2. agent-memory-sniffer analysed entries, produced a plan (1 LLM call)')
print( '  3. GitHub Issue created (simulated), assigned for review')
print( '  4. Auto-remove: stale tactical entries deleted')
print( '  5. Flagged entries surfaced for your decision')
print(f'  6. Memory: {pct:.1%} → {new_pct:.1%}  (freed {freed} chars)')
print()
print('  At the core: GitHub project management.')
print('  Counsellor makes it special with the magical wand of AI. 🧙‍♂️')

## 🔌 Swap: Wire Up Real GitHub Issues

The simulated output above is identical in structure to what a real GitHub Issue would look like.
To make it real, replace the print block in **Cell 11** with:

```python
# pip install PyGithub
from github import Github
import os

g     = Github(os.environ['GITHUB_TOKEN'])   # set in Colab Secrets
repo  = g.get_repo('your-username/counsellor-kba')
issue = repo.create_issue(
    title    = report['title'],
    body     = issue_body,
    assignees= ['your-username'],
    labels   = ['memory-management', 'counsellor-auto'],
)
print(f'Issue created: {issue.html_url}')
```

Everything else stays the same. The Law, the sniffer, the cleanup logic — none of it changes.
GitHub is just the delivery mechanism.

---

*Read the full post: [Decision Archaeology — Why Decisions Die and How a KBA Fixes That](https://sriharshacr.github.io/blogs/decision-archaeology-kba/)*